In [1]:

import re


nodes = 15
counts = [0 for _ in range(nodes)]
counts_to_last = [0 for _ in range(nodes)]
counts_to_share = [0 for _ in range(nodes)]

# this is the pattern to look for when a model is in aggregation and actively sending model data over to a peer.
# when Weighting 0 it means that it is instead sharing the model to someone as is not actually in aggregation
model_transfer_overhead_pattern = r"\[recieve_model\] Recieving Model Data: Length 1957 and Weighting ([1-9]|1[0-5])$"
model_transfer_overhead_pattern_15 = r"\[recieve_model\] Recieving Model Data: Length 1957 and Weighting 15$"
model_transfer_overhead_pattern_sharing = r"\[send_model\] Sending Model Data: Length 1957 and Weighting 0$"

for i in range(nodes):
    file = open(f"./podman-logs/15-nodes/node_{i+1}_logs.txt")
    for line in file:
        if re.search(model_transfer_overhead_pattern,line.strip()):
            counts[i] += 1
        if re.search(model_transfer_overhead_pattern_15,line.strip()):
            counts_to_last[i] += 1
        if re.search(model_transfer_overhead_pattern_sharing,line.strip()):
            counts_to_share[i] += 1
    file.close()

tally = 0
for count in counts:
    tally += count
print(f"Models recieved by each node: {counts}")
print(f"Models recieved Total: {tally}")
tally_last = 0
for count in counts_to_last:
    tally_last += count
print(f"Each time node has been last: {counts_to_last}")
print(f"Time it has Aggregated to finish: {tally_last}")
tally_share = 0
for count in counts_to_share:
    tally_share += count
print(f"Amount of times the node has shared {counts_to_share}")
print(f"Total shares {tally_share}")

Models recieved by each node: [18, 21, 21, 17, 23, 13, 17, 17, 18, 12, 19, 15, 16, 31, 22]
Models recieved Total: 280
Each time node has been last: [1, 1, 0, 0, 1, 0, 1, 0, 2, 0, 1, 0, 0, 1, 2]
Time it has Aggregated to finish: 10
Amount of times the node has shared [28, 14, 1, 1, 15, 0, 14, 2, 28, 0, 14, 0, 1, 15, 30]
Total shares 163


In [2]:
from numpy import average

biggest_agg = max(counts)
averages_agg = average(counts)
minmal_agg = min(counts)
biggest_sh = max(counts_to_share)
averages_sh = average(counts_to_share)
minmal_sh = min(counts_to_share)
print(f"maximum model transfers of a peer aggregating : {biggest_agg}")
print(f"average model transfers of a peer aggregating : {averages_agg}")
print(f"minimum model transfers of a peer aggregating : {minmal_agg}")
print(f"maximum model transfers of a peer sharing : {biggest_sh}")
print(f"average model transfers of a peer sharing : {averages_sh}")
print(f"minimum model transfers of a peer sharing : {minmal_sh}")

maximum model transfers of a peer aggregating : 31
average model transfers of a peer aggregating : 18.666666666666668
minimum model transfers of a peer aggregating : 12
maximum model transfers of a peer sharing : 30
average model transfers of a peer sharing : 10.866666666666667
minimum model transfers of a peer sharing : 0


In [3]:
# the amount of communication rounds is 15 for 15 epochs where each communcation round is 1 epochs
# i was only able to get the records of 14 rounds on the podman logs
rounds = 10

print(f"maximum model transfers per round (agg) : {biggest_agg/rounds}")
print(f"average model transfers per round (agg) : {averages_agg/rounds}")
print(f"minimum model transfers per round (agg) : {minmal_agg/rounds}")
print(f"maximum model transfers per round (sh) : {biggest_sh/rounds}")
print(f"average model transfers per round (sh) : {averages_sh/rounds}")
print(f"minimum model transfers per round (sh) : {minmal_sh/rounds}")

maximum model transfers per round (agg) : 3.1
average model transfers per round (agg) : 1.8666666666666667
minimum model transfers per round (agg) : 1.2
maximum model transfers per round (sh) : 3.0
average model transfers per round (sh) : 1.0866666666666667
minimum model transfers per round (sh) : 0.0


In [4]:
from math import log

# the 2*log2(10) target
# why 2*log2(10)? because it should account for both aggregation being log2(10) AND sharing being log2(10)
print(f"Olog(N) target max communication overhead : {log(nodes,2)}")

Olog(N) target max communication overhead : 3.9068905956085187


In [5]:
print(f"Under O(LogN) (agg) : {biggest_agg/rounds<log(nodes,2)}")
print(f"Under O(LogN) (sh) : {biggest_sh/rounds<log(nodes,2)}")
print(f"Under O(LogN) (agg) :\n actual max: {biggest_agg/rounds} theoritical max: {log(nodes,2)}")
print(f"Under O(LogN) (sh) :\n actual max: {biggest_sh/rounds} theoritical max: {log(nodes,2)}")

Under O(LogN) (agg) : True
Under O(LogN) (sh) : True
Under O(LogN) (agg) :
 actual max: 3.1 theoritical max: 3.9068905956085187
Under O(LogN) (sh) :
 actual max: 3.0 theoritical max: 3.9068905956085187
